In [1]:
from e14c.web import web_search
queries = [
    '"Design and Impact Response of 3D-Printable Tensegrity-Inspired Structures"',
    '"Prestrain-induced bandgap tuning in 3D-printed tensegrity-inspired lattice structures"',
    '"Tensegrity Metamaterials: Toward Failure-Resistant Engineering Systems"',
    '"Fabrication and experimental characterisation of a bistable tensegrity-like unit"',
    '"Accelerated Design of Architected Materials with Multifidelity Bayesian Optimization"',
    '"Toward a novel energy-dissipation metamaterial with tensegrity architecture"',
    '"Experimental investigations on mechanical properties of 3D-printed tensegrity-inspired metamaterials"',
    '"High strain rate response of 3D-printable tensegrity-inspired structures"',
    '"Dynamic analysis of additively manufactured tensegrity structures"',
    '"Integrated fabrication and validation of tensegrity-inspired rigid-flexible mechanical metamaterials"'
]
results = {}
for q in queries:
    results[q] = await web_search(q, category='scholarly', num_results=5)
for q, rr in results.items():
    print('\nQUERY', q)
    for r in rr[:3]:
        print(r.get('title'), '|', r.get('url'), '|', r.get('snippet','')[:400].replace('\n',' '))


QUERY "Design and Impact Response of 3D-Printable Tensegrity-Inspired Structures"
Design and impact response of 3D-printable tensegrity-inspired structures | https://doi.org/10.1016/j.matdes.2019.107966 | # Design and impact response of 3D-printable tensegrity-inspired structures ... # Design and impact response of 3D-printable tensegrity-inspired structures ... We present the design of a 3Dprintable structure with comparable response characteristics as a pinjointed tensegrity structure. • Drop weight simulations and experiments show unique and desirable impact characteristics. • Theoretical studie
Prestrain-induced bandgap tuning in 3D-printed tensegrity-inspired lattice structures | https://doi.org/10.1016/j.eml.2021.101236 | # Prestrain-induced bandgap tuning in 3D-printed tensegrity-inspired lattice structures ... # Prestrain-induced bandgap tuning in 3D-printed tensegrity-inspired lattice structures ... In this letter, we provide experimental evidence of bandgap tunability with g

In [2]:
import pandas as pd, re, pathlib, numpy as np
p=pathlib.Path('/workspace/manuscript-body.tex')
tex=p.read_text()
abstract=re.search(r'\\begin\{abstract\}(.*?)\\todo', tex, flags=re.S).group(1)
plain=re.sub(r'\\[A-Za-z]+(?:\[[^]]*\])?(?:\{([^{}]*)\})?', lambda m: m.group(1) or '', abstract)
plain=re.sub(r'[$~{}\\]', ' ', plain)
words=re.findall(r"\b[\w]+(?:[-–][\w]+)*\b", plain)
print('Abstract whitespace-like token count:', len(words))
print('First/last:', words[:8], words[-8:])
df=pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
print('\nRows:',len(df),'mapped:',df.spec.notna().sum(),'t180 > 1:',(df.t180_mean>1).sum(), 't180 < 1:',(df.t180_mean<1).sum())
print('range',df.t180_mean.min(),df.t180_mean.max(),'span',df.t180_mean.max()-df.t180_mean.min())
print(df[['specimen','spec','t180_mean','e_rebound_mean','mass_g']].to_string(index=False))
print('\nSuggested rows:',len(pd.read_csv('/workspace/t3-prism-bo-suggestions-round1.csv')))

Abstract whitespace-like token count: 190
First/last: ['Tensegrity-inspired', 'architectures', 'combine', 'rigid', 'compression', 'members', 'with', 'a'] ['whether', 'learned', 'design', 'rankings', 'transfer', 'across', 'material', 'systems']

Rows: 8 mapped: 7 t180 > 1: 5 t180 < 1: 3
range 0.8930777877843858 1.0616197738255833 span 0.1685419860411974
specimen spec  t180_mean  e_rebound_mean  mass_g
  6lhxfy   01   0.893078        0.050376   18.50
  6nheas   05   0.997008        0.040246   21.73
  9hhbkp   00   1.018336        0.021501   21.62
  amdjwm  NaN   0.980495        0.029618     NaN
  autv5r   02   1.040430        0.026810   22.04
  bag26v   08   1.061620        0.024098   21.42
  bpx68c   S0   1.011072        0.020442   20.23
  nvxsrv   04   1.027549        0.026564   20.66

Suggested rows: 9


In [3]:
from e14c.web import fetch_paper
import asyncio
papers = {
'Pajunen2019':'10.1016/j.matdes.2019.107966',
'Pajunen2021':'10.1016/j.eml.2021.101236',
'Bauer2021':'10.1002/adma.202005647',
'Intrigila2022':'10.1016/j.addma.2022.102946',
'Mo2023':'10.1061/JENMDT.EMENG-7033',
'Santos2023':'10.1002/adma.202300639',
'Sabouni2024':'10.24425/ace.2024.150987',
'Almeida2025':'10.1016/j.ijsolstr.2025.113590',
'Davami2025':'10.1016/j.ijimpeng.2024.105208',
'Wang2026':'10.1016/j.addma.2026.105107'}
async def get(k,d):
    try:
        x=await fetch_paper(doi=d, multimodal=False)
        return k,x,None
    except Exception as e:return k,None,repr(e)
out=await asyncio.gather(*(get(k,d) for k,d in papers.items()))
fetched={}
for k,x,e in out:
    if x:
        text='\n'.join(pg.get('text','') for pg in x.get('pages',[]))
        fetched[k]=text
        print(k,'OK pages',len(x.get('pages',[])),'chars',len(text))
    else: print(k,'FAIL',e[:180])

Pajunen2019 OK pages 9 chars 71153
Pajunen2021 OK pages 7 chars 48629
Bauer2021 FAIL HTTPStatusError("Client error '403 Forbidden' for url 'https://prod.api.paperqa.app/api/paper'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status
Intrigila2022 OK pages 13 chars 76118
Mo2023 FAIL HTTPStatusError("Client error '403 Forbidden' for url 'https://prod.api.paperqa.app/api/paper'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status
Santos2023 OK pages 10 chars 207658
Sabouni2024 OK pages 16 chars 51149
Almeida2025 FAIL HTTPStatusError("Client error '403 Forbidden' for url 'https://prod.api.paperqa.app/api/paper'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status
Davami2025 FAIL HTTPStatusError("Client error '403 Forbidden' for url 'https://prod.api.paperqa.app/api/paper'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status
Wang2026 FAIL HTTPStatusError("Clien

In [4]:
terms=['bayesian','optimization','multi-material','multimaterial','impact','drop','experiment','physical','finite element','simulation','prestress','pretension']
for k,text in fetched.items():
    low=text.lower()
    print('\n###',k)
    for term in terms:
        print(f'{term}: {low.count(term)}',end=' | ')
    print()
    # show abstract-ish first occurrences around Bayesian/optimization/multimaterial
    for term in ['bayesian','optimization','multi-material','multimaterial']:
        i=low.find(term)
        if i>=0:
            s=re.sub(r'\s+',' ',text[max(0,i-250):i+450])
            print(term,':',s[:700])


### Pajunen2019
bayesian: 0 | optimization: 2 | multi-material: 0 | multimaterial: 0 | impact: 81 | drop: 11 | experiment: 37 | physical: 0 | finite element: 0 | simulation: 40 | prestress: 17 | pretension: 0 | 
optimization : Resources, Visualization, work elastically deform and are reusable under multiple impacts. Supervision, Project administration, Funding acquisition. This shows that, even without optimization of energy absorption, our design shows novel characteristics, making it promising for use in Acknowledgmentsultra-lightweight, energy absorbing metamaterials. This research was conducted with Government support under 4. Conclusions and awarded by DoD, Air Force

### Pajunen2021
bayesian: 0 | optimization: 1 | multi-material: 0 | multimaterial: 0 | impact: 2 | drop: 0 | experiment: 45 | physical: 6 | finite element: 0 | simulation: 5 | prestress: 2 | pretension: 0 | 
optimization : periodic elastomeric structures, International Journal [38] J. Zhang, M. Ohsaki, Tensegrity St

In [5]:
from e14c.web import web_search
qs={
'JMD abstract':'ASME Journal of Mechanical Design author guidelines abstract 150 200 words',
'Bauer':'10.1002/adma.202005647 additive manufacturing experiment optimization',
'Mo':'10.1061/JENMDT.EMENG-7033 multifidelity Bayesian optimization simulation experimental ground truth',
'Almeida':'10.1016/j.ijsolstr.2025.113590 high strain rate 3D printable tensegrity experiment',
'Davami':'10.1016/j.ijimpeng.2024.105208 dynamic analysis additively manufactured tensegrity experiment',
'Wang':'10.1016/j.addma.2026.105107 dual-nozzle Bayesian optimization impact'
}
for name,q in qs.items():
    rr=await web_search(q, num_results=6)
    print('\n###',name)
    for r in rr[:4]:
        print('-',r.get('title'),'|',r.get('url'))
        print(' ',re.sub(r'\s+',' ',r.get('snippet',''))[:650])


### JMD abstract
- Submission Instructions - ASME JMD | https://asmejmd.org/resources-2/submission-instructions/
  The paper abstract should be 150-200 words long using only text with Latin characters (no special characters, Greek, or math symbols). The purposes of the abstract are: (1) To give a clear indication of the objective, scope, and results so that readers may determine whether the full text will be of particular interest to them; and (2) To provide key words and phrases for indexing, abstracting, and retrieval purposes. The abstract text should be organized to include the following categories in the order noted: Background, method of approach, results, and conclusions.
- Title and Abstract - ASME JMD | https://asmejmd.org/resources-2/title-and-abstract/
  The paper abstract should be 150-200 words long using only text with Latin characters (no special characters, Greek, or math symbols). The purposes of the abstract are: (1) To give a clear indication of the objective, scope


### Bauer
- Tensegrity Metamaterials: Toward Failure-Resistant Engineering Systems through Delocalized Deformation | https://onlinelibrary.wiley.com/doi/10.1002/adma.202005647
  Login / Register * **[Individual login](https://onlinelibrary.wiley.com/action/showLogin?acdl-redirect=true&uri=/doi/10.1002/adma.202005647) * **[Institutional login](https://onlinelibrary.wiley.com/action/ssostart?redirectUri=/doi/10.1002/adma.202005647) * [REGISTER](https://onlinelibrary.wiley.com/action/registration?redirectUri=/doi/10.1002/adma.202005647&acdl-redirect=true) [Advanced Materials](https://onlinelibrary.wiley.com/journal/15214095) ... [Volume 33, Issue 10](https://onlinelibrary.wiley.com/toc/15214095/2021/33/10)2005647[![Advanced Materials](https://onlinelibrary.wiley.com/pb-assets/journal-banners/15214095-1583485537800.jpg)]
- Process optimization of complex geometries using feed forward control for laser powder bed fusion additive manufacturing | https://www.sciencedirect.com/science/article


### Mo
- Accelerated Design of Architected Materials with Multifidelity Bayesian Optimization (Journal Article) | OSTI.GOV | https://www.osti.gov/biblio/2420964
  # Accelerated Design of Architected Materials with Multifidelity Bayesian Optimization ... 01 June 2023 · Journal of Engineering Mechanics ... DOI: https://doi.org/10.1061/jenmdt.emeng-7033· OSTI ID:2420964 ... | Using simulation to accelerate autonomous experimentation: A case study using mechanics Gongora, Aldair E.; Snapp, Kelsey L.; Whiting, Emily iScience, Vol. 24, Issue 4 https://doi.org/10.1016/j.isci.2021.1 ... 2262 | journal | ... 2021 | ... | A Bayesian experimental autonomous researcher for mechanical design Gongora, Aldair E.; Xu, Bowen; Perry, Wyatt Science Advances, Vol. 6, Issue 15 https://doi.org/10.1126/sciadv.aaz1708 | journ
- Accelerated Design of Architected Materials with Multifidelity Bayesian Optimization | NSF Public Access Repository | https://par.nsf.gov/biblio/10414976
  Accelerated Design of Archi


### Almeida
- High strain rate response of 3D-printable tensegrity- ... | https://www.sciencedirect.com/science/article/abs/pii/S0020768325003762
  High strain rate response of 3D-printable tensegrity-inspired structures - ScienceDirect ... [Volume 322](https://www.sciencedirect.com/journal/international-journal-of-solids-and-structures/vol/322/suppl/C),1 November 2025, 113590 ... # High strain rate response of 3D-printable tensegrity-inspired structures ... [https://doi.org/10.1016/j.ijsolstr.2025.113590](https://doi.org/10.1016/j.ijsolstr.2025.113590)[Get rights and content](https://s100.copyright.com/AppDispatchServlet?publisherName=ELS&contentID=S0020768325003762&orderBeanReset=true) ... The potential of using tensegrity structures as architected lattices or metamaterials is promisin
- Nara Almeida (0000-0003-2336-8644) - ORCID | https://orcid.org/0000-0003-2336-8644
  #### High strain rate response of 3D-printable tensegrity-inspired structures ... International Journal of Solids


### Davami
- Dynamic Analysis of Additively Manufactured Tensegrity ... | https://scholar.afit.edu/cgi/viewcontent.cgi?article=2665&context=facpub
  Dynamic Analysis of Additively Manufactured Tensegrity Structures ... Recommended Citation Davami, K., Rowe, R., Gulledge, B., Park, J., Beheshti, A., Palazotto, A., Tavangarian, F., & Beck, S. (2025). Dynamic analysis of additively manufactured tensegrity structures. International Journal of Impact Engineering, 198, 105208. https://doi.org/10.1016/j.ijimpeng.2024.105208 This Article is brought to you for free and open access by AFIT Scholar. It has been accepted for inclusion in Faculty Publications by an authorized administrator of AFIT Scholar. For more information, please contact AFIT.ENWL.Repository@us.af.mil. ... Analysis of Add ... Abs
- Dynamic Analysis of Additively Manufactured Tensegrity ... | https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4785600
  Dynamic Analysis of Additively Manufactured Tensegrity Structures by Keiva


### Wang
- Physics-constrained multi-objective bayesian optimization to accelerate 3d printing of thermoplastics | NSF Public Access Repository | https://par.nsf.gov/biblio/10590231
  Title: Physics-constrained multi-objective bayesian optimization to accelerate 3d printing of thermoplastics ... Lin, Jian Publisher / Repository: ScienceDirect Date Published: 2024-04-01 Journal Name: Additive Manufacturing Volume: 86 Issue: C ISSN: 2214-8604 Page Range / eLocation ID: 104204 Format(s): Medium: X Sponsoring Org: National Science Foundation ... - Free Publicly Accessible Full Text - Accepted Manuscript - Journal Article: - https://doi.org/10.1016/j.addma.2024.104204
- Physics-constrained multi-objective bayesian optimization to accelerate 3d printing of thermoplastics | NSF Public Access Repository | https://par.nsf.gov/biblio/10590231-physics-constrained-multi-objective-bayesian-optimization-accelerate-printing-thermoplastics
  Title: Physics-constrained multi-objective bayesian optimiz

In [6]:
# Extract concise evidence windows from retrieved full texts for novelty-table accuracy.
for k in ['Pajunen2019','Intrigila2022','Pajunen2021','Santos2023','Sabouni2024']:
    text=fetched[k]
    low=text.lower()
    print('\n###',k)
    for term in ['abstract','drop weight','impact test','quasi-static','multi-material','bayesian optimization']:
        i=low.find(term)
        if i>=0:
            print(term, '=>', re.sub(r'\s+',' ',text[max(0,i-180):i+520])[:700])
# Save evidence notes in a simple dataframe for reproducibility.
rows=[]
for k,text in fetched.items():
    low=text.lower()
    rows.append({'study':k,'bayesian_mentions':low.count('bayesian'),'multimaterial_mentions':low.count('multi-material')+low.count('multimaterial'),'impact_mentions':low.count('impact'),'drop_mentions':low.count('drop')})
evidence=pd.DataFrame(rows)
print('\n',evidence.to_string(index=False))


### Pajunen2019
drop weight =>  I C A L A B S T R A C T • We present the design of a 3D- printable structure with compara- ble response characteristics as a pin- jointed tensegrity structure. • Drop weight simulations and exper- iments show unique and desirable impact characteristics. • Theoretical studies on tenseg- rity structures with elastically buckling struts are corroborated experimentally for the ﬁrst time. • These studies pave the way for new research in the area of manufac- turable tensegrity-inspired metama- terials. A R T I C L E I N F O A B S T R A C T Article history: Recent studies demonstrate the potential 
impact test =>  equivalent mechanical response. Numerical simulations inform quasi-static compression experiments and Buckling dynamic drop weight impact tests. The structure’s responses correspond well to the pin-jointed tenseg- Architected unit cells rity, exhibiting desirable characteristics such as post-buckling stability, resilience under severe deformation, hi